# Six-time S2 legacy-exact rebuild

This notebook starts from the frozen 4450-row source table. It uses 4407 all-CDSE-raw rows and 43 rows whose seasonal/year inputs come from the old engg-leung 512 dataset. It contains no download or pull stage.

In [ ]:
from pathlib import Path
import json
import subprocess

PYTHON = '/home/yuyao/miniconda3/envs/panopticon/bin/python'
PIPELINE = '/home/yuyao/panopticon/Upgraded_dataset/s2_6time_cdse_legacy512_rebuild.py'
CSV_ROOT = Path('/home/yuyao/methane_train/Upgrade_data_pipeline/csv/s2_6time_cdse_legacy512_exact')
OUT_512 = Path('/mnt/engg-niulab/yuyao/preprocessed_512/S2_6time_cdse_legacy512_exact')
OUT_32 = Path('/mnt/engg-niulab/yuyao/final_crop/s2_6time_cdse_legacy512_exact_32')
OUT_224 = Path('/mnt/engg-niulab/yuyao/final_crop/s2_6time_cdse_legacy512_exact_32_to_224')
SOURCE_CSV = CSV_ROOT / 's2_6time_cdse_legacy512_sources.csv'
COMPLETE_512 = CSV_ROOT / 's2_6time_cdse_legacy512_512_complete.csv'
SPLIT_ROOT = CSV_ROOT / 'temporal_split'

In [ ]:
# Cell 4 equivalent, part 1: freeze and validate every six-time source path.
subprocess.run([
    PYTHON, PIPELINE, 'build-manifest',
    '--out-512-root', str(OUT_512),
    '--source-csv', str(SOURCE_CSV),
    '--source-audit-json', str(CSV_ROOT / 's2_6time_cdse_legacy512_sources.audit.json'),
    '--stat-workers', '32',
], check=True)

In [ ]:
# Cell 4 equivalent, part 2: create six 512 images and rebuild the exact old mask.
# reuse-validated avoids rewriting image arrays already verified against Cell 4.
# Change it to standardize to rerun Cell 4 directly from raw/legacy inputs.
subprocess.run([
    PYTHON, PIPELINE, 'build-512',
    '--source-csv', str(SOURCE_CSV),
    '--qa-csv', str(CSV_ROOT / 's2_6time_cdse_legacy512_512_qa.csv'),
    '--complete-csv', str(COMPLETE_512),
    '--image-mode', 'reuse-validated',
    '--workers', '16',
], check=True)

In [ ]:
# Old split cell, corrected to keep every event_group_id wholly on one side.
subprocess.run([
    PYTHON, PIPELINE, 'split',
    '--complete-csv', str(COMPLETE_512),
    '--split-root', str(SPLIT_ROOT),
    '--target-ratio', '0.85', '--min-ratio', '0.80', '--max-ratio', '0.90',
], check=True)
split_audit = json.loads((SPLIT_ROOT / 'split_audit.json').read_text())
TRAIN_512 = Path(split_audit['train_csv'])
TEST_512 = Path(split_audit['test_csv'])
split_audit

In [ ]:
# Old 32-pixel crop cell copied to all six timepoints.
# Per plume: 16 crops containing the center 20x20 box, plus 16 random crops.
subprocess.run([
    PYTHON, PIPELINE, 'crop-32',
    '--train-csv', str(TRAIN_512), '--test-csv', str(TEST_512),
    '--out-32-root', str(OUT_32),
    '--workers', '16', '--resume',
], check=True)

In [ ]:
# Old bilinear 32-to-224 image resize, now applied to all six image stacks.
# plume.tif remains the correct 32x32 crop, matching the old notebook.
subprocess.run([
    PYTHON, PIPELINE, 'resize-224',
    '--out-32-root', str(OUT_32), '--out-224-root', str(OUT_224),
    '--workers', '8', '--batch-files', '512',
], check=True)

In [ ]:
# Final hard audit: event/plume leakage, labels, masks, timestamps, and paths.
subprocess.run([
    PYTHON, PIPELINE, 'audit',
    '--train-csv', str(OUT_224 / 'train_patches_224.csv'),
    '--test-csv', str(OUT_224 / 'test_patches_224.csv'),
    '--audit-json', str(OUT_224 / 'dataset_audit.json'),
    '--path-audit-rows', '1000', '--path-stat-workers', '32',
], check=True)
json.loads((OUT_224 / 'dataset_audit.json').read_text())

In [ ]:
# Full-backbone training; mean/std are recomputed from the final train CSV.
print('bash /home/yuyao/panopticon/Upgraded_dataset/train_s2_6time_cdse_legacy512_exact.sh')